# SkillOpt: GEPA Skill Optimization (Static & Eval-Based)

This notebook runs the same pipelines as the CLI scripts:

| Script | CLI | Mode |
|--------|-----|------|
| `scripts/optimize_skill.py` | `python main.py optimize` | **Static-only** — 100% heuristic metric |
| `scripts/optimize_skill_with_evals.py` | `python main.py optimize-evals` | **Eval-based** — 40% static + 60% LLM-as-judge |

## Script analysis

### `optimize_skill.py` (static)

- **GEPA seed**: `SKILL.md` content (truncated to `content_limit`, default 8000 chars)
- **Evaluator** (`create_skill_evaluator`): returns `(score, feedback)` per candidate
  - 25% filler phrase removal
  - 20% conciseness (target 40–70% size reduction)
  - 25% code block preservation
  - 15% structure (frontmatter, `##` sections, code fences)
  - 15% verbose pattern removal
- **Output**: `<skill>_GEPA_Optimized/SKILL.md`

### `optimize_skill_with_evals.py` (eval-based)

Three-phase pipeline:

1. **Phase 1** — Load `evals.json`, or `--generate-evals`; auto-generate `expectations` for cases missing them
2. **Phase 2** — GEPA with hybrid evaluator: 40% `static_analysis_score` + 60% `judge_skill_against_assertions` (LLM-as-judge per eval case)
3. **Phase 3** — Final validation, `benchmark.json` with per-assertion PASS/FAIL

- **Output**: `<skill>_GEPA_Eval_Optimized/SKILL.md` + `benchmark.json`

## How to use this notebook

1. Set `MODE` to `"static"` or `"eval"` in the configuration cell
2. Set `SKILL_PATH` (and `EVALS_PATH` / `GENERATE_EVALS` for eval mode)
3. Run cells top to bottom
4. Set `RUN_OPTIMIZATION = False` to analyze only (no API calls for GEPA)

## 1. Setup

In [ ]:
# Install once if needed:
# !uv sync
# or: !pip install -e . gepa openai python-dotenv

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

from skillopt import SkillParser, SkillAnalyzer

# Repo root (works when cwd is repo root or notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.optimize_skill import optimize_skill
from scripts.optimize_skill_with_evals import optimize_skill_with_evals

load_dotenv(ROOT / ".env")

parser = SkillParser()
analyzer = SkillAnalyzer()

print(f"Repo root: {ROOT}")
print("Imports OK: optimize_skill, optimize_skill_with_evals")

## 2. Configuration

Set **`MODE`** to `"static"` or `"eval"`, then adjust paths and model settings.

In [ ]:
# ---- CONFIGURE ----
MODE = "eval"  # "static" | "eval"

SKILL_PATH = ROOT / "examples/example_2/dad-joke"

# Eval mode only:
EVALS_PATH = ROOT / "examples/example_2/dad-joke/evals_with_assertions.json"  # or None
GENERATE_EVALS = False  # True if EVALS_PATH is None/missing

OUTPUT_DIR = None  # None → script default suffix (_GEPA_Optimized / _GEPA_Eval_Optimized)

MODEL = os.getenv("SKILLOPT_MODEL", "openai/gpt-4o")
API_KEY = os.getenv("OPENAI_API_KEY") or os.getenv("GEMINI_API_KEY") or os.getenv("API_KEY")
API_BASE = os.getenv("OPENAI_API_BASE")  # e.g. http://localhost:11434/v1 for Ollama

MAX_METRIC_CALLS = 10
CONTENT_LIMIT = 8000  # static mode only
RUN_OPTIMIZATION = True  # False = analyze only (dry run)
# -------------------

if MODE not in ("static", "eval"):
    raise ValueError(f"MODE must be 'static' or 'eval', got: {MODE!r}")

if not API_KEY and RUN_OPTIMIZATION:
    raise ValueError(
        "No API key. Set OPENAI_API_KEY, GEMINI_API_KEY, or API_KEY in .env "
        "(or set RUN_OPTIMIZATION=False for analysis only)."
    )

print(f"Mode: {MODE}")
print(f"Skill: {SKILL_PATH}")
print(f"Model: {MODEL}")
print(f"Run optimization: {RUN_OPTIMIZATION}")
if MODE == "eval":
    print(f"Evals: {EVALS_PATH or '(auto-generate)' if GENERATE_EVALS else 'required'}")

## 3. Load & Analyze (shared)

In [ ]:
if not SKILL_PATH.exists():
    raise FileNotFoundError(f"Skill not found: {SKILL_PATH}")

skill = parser.parse_directory(SKILL_PATH)
report = analyzer.analyze(skill)

errors = [i for i in report.issues if i.severity == "error"]
warnings = [i for i in report.issues if i.severity == "warning"]
suggestions = [i for i in report.issues if i.severity == "suggestion"]

print(f"Loaded: {skill.name}")
print(f"  Lines: {skill.total_lines}")
print(f"  References: {len(skill.references)}")
print(f"\nAnalysis: {report.score}/100")
print(f"  Errors: {len(errors)}, Warnings: {len(warnings)}, Suggestions: {len(suggestions)}")
print()
for issue in report.issues[:12]:
    icon = {"error": "[ERR]", "warning": "[WRN]", "suggestion": "[SUG]"}.get(issue.severity, "[?]")
    print(f"  {icon} {issue.category}: {issue.message}")
if len(report.issues) > 12:
    print(f"  ... and {len(report.issues) - 12} more")

## 4. Run optimization

Calls `optimize_skill()` when `MODE == "static"`, or `optimize_skill_with_evals()` when `MODE == "eval"`.

These are the same functions used by `scripts/optimize_skill.py` and `scripts/optimize_skill_with_evals.py`.

In [ ]:
if MODE == "static":
    default_out = SKILL_PATH.parent / f"{SKILL_PATH.name}_GEPA_Optimized"
    out = Path(OUTPUT_DIR) if OUTPUT_DIR else default_out

    print("=" * 60)
    print("STATIC-ONLY OPTIMIZATION (optimize_skill.py)")
    print("=" * 60)

    result = optimize_skill(
        skill_path=SKILL_PATH,
        output_path=out if RUN_OPTIMIZATION else None,
        model=MODEL,
        max_evals=MAX_METRIC_CALLS,
        content_limit=CONTENT_LIMIT,
        dry_run=not RUN_OPTIMIZATION,
        verbose=True,
        api_key=API_KEY,
        api_base=API_BASE,
    )
else:
    default_out = SKILL_PATH.parent / f"{SKILL_PATH.name}_GEPA_Eval_Optimized"
    out = Path(OUTPUT_DIR) if OUTPUT_DIR else default_out

    evals_path = Path(EVALS_PATH) if EVALS_PATH else None
    if evals_path and not evals_path.exists():
        evals_path = None

    print("=" * 60)
    print("EVAL-BASED OPTIMIZATION (optimize_skill_with_evals.py)")
    print("=" * 60)

    result = optimize_skill_with_evals(
        skill_path=SKILL_PATH,
        evals_path=evals_path,
        output_path=out if RUN_OPTIMIZATION else None,
        model=MODEL,
        max_evals=MAX_METRIC_CALLS,
        dry_run=not RUN_OPTIMIZATION,
        generate_evals=GENERATE_EVALS or evals_path is None,
        verbose=True,
        api_key=API_KEY,
        api_base=API_BASE,
    )

## 5. Results

In [ ]:
if result.get("dry_run"):
    print("Dry run — no optimized SKILL.md produced.")
    if MODE == "eval" and "evals" in result:
        print(f"Eval cases prepared: {len(result['evals'].get('evals', []))}")
elif not result.get("optimized_content"):
    print("No optimized content in result.")
else:
    optimized = result["optimized_content"]
    original_len = result["original_length"]
    optimized_len = result["optimized_length"]

    print(f"Mode: {MODE}")
    print(f"Original:  {original_len:,} chars")
    print(f"Optimized: {optimized_len:,} chars")
    print(f"Reduction: {result['reduction_percent']:.1f}%")
    print(f"Code blocks: {result['optimized_code_blocks']}/{result['original_code_blocks']}")

    if MODE == "static":
        print(f"Metric score: {result['metric_score']:.3f}")
    else:
        print(f"Static score:     {result['static_score']:.3f}")
        print(f"Assertion pass:   {result['assertion_score']:.3f}")
        print(f"Combined (40/60): {result['combined_score']:.3f}")
        print("\nPer-eval assertion results:")
        for r in result.get("assertion_results", []):
            status = "PASS" if r["pass_rate"] == 1.0 else "PARTIAL" if r["pass_rate"] > 0 else "FAIL"
            print(f"  [{r['eval_id']}] {r['eval_name']}: {r['passed']}/{r['total']} ({r['pass_rate']:.0%}) {status}")

    if result.get("output_path"):
        print(f"\nSaved to: {result['output_path']}")
        if MODE == "eval":
            bench = result["output_path"] / "benchmark.json"
            if bench.exists():
                print(f"  benchmark.json: {bench}")

In [ ]:
# Preview optimized SKILL.md (first 2500 chars)
if result.get("optimized_content"):
    text = result["optimized_content"]
    print("Optimized SKILL.md preview")
    print("=" * 60)
    print(text[:2500])
    if len(text) > 2500:
        print("\n... (truncated)")

In [ ]:
# Load benchmark.json in eval mode
if MODE == "eval" and result.get("output_path"):
    bench_path = Path(result["output_path"]) / "benchmark.json"
    if bench_path.exists():
        with open(bench_path) as f:
            benchmark = json.load(f)
        print("benchmark.json scores:")
        print(json.dumps(benchmark.get("scores", {}), indent=2))

## 6. Switch mode & re-run

To compare both approaches on the same skill:

1. Run with `MODE = "static"`, then inspect `*_GEPA_Optimized/SKILL.md`
2. Change to `MODE = "eval"`, set `EVALS_PATH` or `GENERATE_EVALS = True`, re-run from section 2

Equivalent CLI commands:

```bash
# Static
uv run python main.py optimize examples/example_2/dad-joke --model openai/gpt-4o

# Eval-based
uv run python main.py optimize-evals examples/example_2/dad-joke \
  --evals examples/example_2/dad-joke/evals_with_assertions.json
```